In [3]:
from bs4 import BeautifulSoup
import requests

In [4]:
url = 'https://www.canada.ca/fr/emploi-developpement-social/programmes/assurance-emploi/ae-liste/rapports/guide.html'
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')

list_parent = soup.find('div', class_='panel-body')

list_items = list_parent.find_all('li')

chapter_list = []

for item in list_items:
    chapter_list.append(item.get_text(strip=True))

In [16]:
sections = ['1.1.2', '5.6.2.1', '22.2.8']
for number in sections:
    first_number = number.split('.')[0]

1
5
22


In [5]:
first_number = 10
section_chapter = chapter_list[first_number - 1]
print(section_chapter)

Chapitre 10 - Disponibilité


In [6]:
digest = open('output_french_full.txt', 'r', encoding='utf-8')
lines = digest.readlines()

section = []
section_title = []
section_text = []
section_chapter = []

for line in lines:
    line = line.split(':', 1)
    if 'section_number' in line[0]:
        section_number = line[1].strip().replace(',','')
        section.append(section_number)
        first_number = int(section_number.split('.')[0])
        chapter = chapter_list[first_number - 1]
        section_chapter.append(chapter)
    elif 'section_title' in line[0]: 
        section_title.append(line[1].strip().replace(',',''))
    elif 'section_text' in line[0]:
        section_text.append(line[1].strip())
    else:
        pass

In [8]:
import json

sections = []
for num, title, text, chapter in zip(section, section_title, section_text, section_chapter):
    section = {
        'section_number': num,
        'section_title': title,
        'section_chapter': chapter,
        'section_text': text
    }
    sections.append(section)

with open('formatted_sections_french_full.json', 'w', encoding='utf-8') as json_file:
    json.dump(sections, json_file, indent=4, ensure_ascii=False)

In [ ]:
import json
import pandas as pd

with open('output_noquotation1.txt', 'r') as file:
    text_data = file.read()

data = json.loads(text_data)

for section in data:
    first_number = section['section_number'].split('.')[0]
    section_chapter = chapter_list[first_number - 1]
    section['section_chapter'] = section_chapter

with open('testchapteroutput.txt', 'w') as file:
    json.dump(data, file, indent=4)

In [ ]:
def extract_meta_from_pages(start_url):
    
    descriptions = []
    keywords = []
    i = 0
    while i < 50:

        response = requests.get(start_url)

        if response.status_code == 200:
            soup = BeautifulSoup(response.text, "html.parser")

            title_parent = soup.find('div', class_='mwstitle section')
            title = title_parent.find("h1").text
            description_meta = soup.find("meta", attrs={"name":"description"})
            keyword_meta = soup.find("meta", attrs={"name":"keywords"})

            if 'Section' in title:

                if description_meta:
                    description_content = description_meta['content']
                    descriptions.append(description_content)

                if keyword_meta:
                    keyword_content = keyword_meta['content']
                    keywords.append(keyword_content)
        
            next_button = soup.find("a", href=True, rel="next")
    
            if next_button:
                start_url = "https://www.canada.ca" + next_button['href']
                i += 1
            else:
                break
        else:
            print("Failed to fetch", start_url)
            break
    return descriptions, keywords

In [ ]:
descriptions, keywords = extract_meta_from_pages('https://www.canada.ca/en/employment-social-development/programs/ei/ei-list/reports/digest/chapter-25/authority.html')

In [ ]:
import csv 

with open('meta_data1.csv', 'a', newline='') as csvfile:
    writer = csv.writer(csvfile)
    
    if csvfile.tell() == 0:
        writer.writerow(['Description', 'Keywords'])
    
    for desc, key in zip(descriptions, keywords):
        writer.writerow([desc, key])